# Convolutional Neural Network

## Introduction to Convolutional Neural Networks

A **convolutional neural network (CNN)** learns spatial patterns in images, from simple edges and textures to more complex shapes. Unlike a fully connected network applied directly to pixels, a convolutional layer uses small, learnable filters shared across image locations. This preserves local spatial relationships and reduces the number of parameters needed to detect a feature throughout an image.

### What is a convolution?

A **convolution** combines an input with a small array of weights called a **kernel**, **filter** or **feature detector**. For an image, imagine sliding the kernel across the pixels: at each location, multiply each kernel weight by the corresponding pixel value, then add the products. Repeating this operation produces a **feature map** showing where a particular pattern occurs. For example, a $3\times3$ kernel with every weight equal to $1/9$ averages a grayscale image's local neighborhoods, creating a blur; other kernels can highlight edges. In a CNN, the kernel weights are learned from training data rather than chosen by hand.

https://sesen.ai/convolution-visualizer

https://adamharley.com/nn_vis/

Strict mathematical convolution flips the kernel before sliding it. CNN libraries such as Keras usually omit this flip, performing **cross-correlation**, but conventionally call the operation convolution. The expression below uses this CNN convention, with stride 1 and no padding; the bias $b$ is added to each weighted sum.

For an input $X$, one filter $K$ produces a feature map with entries:

$$Z_{i,j}=b+\sum_{u,v,c}K_{u,v,c}X_{i+u,j+v,c}$$

where $u,v$ index the filter's spatial positions and $c$ indexes input channels. Multiple filters learn different patterns. The **ReLU** activation:

$$\operatorname{ReLU}(z)=\max(0,z)$$

introduces nonlinearity.

**Max pooling** keeps the largest activation in each local window, reducing spatial dimensions and sensitivity to small shifts. Stacking convolution and pooling layers builds increasingly abstract features. **Flattening** converts the final feature maps into a vector, and **dense layers** combine these features for classification.

For binary classification, a single **sigmoid** output gives the model's estimated probability of class 1:

$$p=\sigma(z)=1/(1+e^{-z})$$

Training adjusts the filters and dense weights through backpropagation to minimize binary cross-entropy:

$$\mathcal{L}=-\frac{1}{N}\sum_{n=1}^{N}[y_n\log p_n+(1-y_n)\log(1-p_n)]$$

where $y_n\in\{0,1\}$ is the true label.

This notebook uses **TensorFlow/Keras** to train a CNN from scratch to distinguish **cats from dogs**, then applies it to a single image.

### 1. Data preprocessing

- Load labeled images from `dataset/training_set` and `dataset/test_set` using `ImageDataGenerator` and `flow_from_directory`; subdirectory names determine the class labels.
- Resize images to $64\times64$ RGB pixels and generate batches of **32** with binary labels.
- Rescale pixel values with $x'=x/255$ for both sets. Augment only training images with shear (`0.2`), zoom (`0.2`), and random horizontal flips to increase training variety and help reduce overfitting.

### 2. CNN architecture

The sequential model contains two convolution/pooling blocks, followed by a dense classifier. Each convolution uses **32 filters**, a $3\times3$ kernel, and ReLU; each max-pooling layer uses a $2\times2$ window with stride 2. With the default valid padding and convolution stride 1, the tensor shapes are:

```mermaid
flowchart TD
    A["RGB image: 64 x 64 x 3"] --> B["Conv2D + ReLU: 62 x 62 x 32"]
    B --> C["Max pooling: 31 x 31 x 32"]
    C --> D["Conv2D + ReLU: 29 x 29 x 32"]
    D --> E["Max pooling: 14 x 14 x 32"]
    E --> F["Flatten: 6272 features"]
    F --> G["Dense: 128 units + ReLU"]
    G --> H["Dense: 1 unit + sigmoid"]
    H --> I["Class-1 probability"]
```

### 3. Compilation and training

- Compile with the **Adam** optimizer, **binary cross-entropy** loss, and **accuracy** as the reported metric.
- Train for **25 epochs**, passing `test_set` as `validation_data` to monitor validation loss and accuracy. Because this set is used during training for validation, it is not a separate, untouched final evaluation set.
- No training outputs or measured accuracy are saved in this notebook, so it does not establish a numerical performance result.

### 4. Single-image prediction

The final example loads `dataset/single_prediction/cat_or_dog_1.jpg`, resizes it to $64\times64$, converts it to a NumPy array, adds a batch dimension to obtain shape $(1,64,64,3)$, and calls `cnn.predict`. It accesses `training_set.class_indices`, maps the result to `dog` or `cat`, and prints the label.

**Prediction caveats:** the example does not divide the single image's pixels by 255, unlike the training and validation pipelines, and it checks whether the sigmoid output is exactly 1 rather than applying a probability threshold. For consistent inference, the image should receive the same rescaling, and a decision rule such as $\hat{y}=1$ when $p\geq0.5$ (otherwise $0$) should be interpreted using `training_set.class_indices`. The existing code assigns `dog` only when the output equals 1 and `cat` otherwise.


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
tf.__version__

'2.21.0'

## Part 1 - Data Preprocessing

* unzip the dataset.zip file into the current working directory, which creates a `dataset` folder with `training_set` and `test_set` subfolders. Each of these contains `cats` and `dogs` subfolders with images.

### Preprocessing the Training set

- preprocess the training images with `ImageDataGenerator` to rescale pixel values (feature scaling) and apply data augmentation (shear, zoom, horizontal flip) in order to reduce overfitting.
- The training images are resized to 64x64 pixels and grouped into batches of 32, with binary labels inferred from the subfolder names.

In [3]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size=(64, 64),
                                                 batch_size=32,
                                                 class_mode='binary')

Found 8000 images belonging to 2 classes.


### Preprocessing the Test set

- we are not doing the same transformations on the test set, only rescaling the pixel values. (we don't touch the test set, we only use it to evaluate the model's performance)
- As the training set, the test images are resized to 64x64 pixels and grouped into batches of 32, with binary labels inferred from the subfolder names.

In [4]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size=(64, 64),
                                            batch_size=32,
                                            class_mode='binary')

Found 2000 images belonging to 2 classes.


## Part 2 - Building the CNN

### Initialising the CNN

In [5]:
cnn = tf.keras.models.Sequential()

### Step 1 - Adding a first convolution layer

In [6]:
cnn.add(tf.keras.Input(shape=(64, 64, 3)))
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))

### Step 2 - And pooling

- pool size: 2x2

In [7]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [8]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [9]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

- add a neuron network layer with 128 neurons and ReLU activation function

In [10]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

In [11]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [12]:
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [13]:
cnn.fit(x=training_set, validation_data=test_set, epochs=25)

Epoch 1/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - accuracy: 0.5964 - loss: 0.6655 - val_accuracy: 0.6605 - val_loss: 0.6182
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.6769 - loss: 0.5966 - val_accuracy: 0.7085 - val_loss: 0.5758
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.7161 - loss: 0.5545 - val_accuracy: 0.7430 - val_loss: 0.5270
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7410 - loss: 0.5172 - val_accuracy: 0.7420 - val_loss: 0.5263
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 52ms/step - accuracy: 0.7574 - loss: 0.5011 - val_accuracy: 0.7540 - val_loss: 0.5074
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 49ms/step - accuracy: 0.7742 - loss: 0.4736 - val_accuracy: 0.7605 - val_loss: 0.4861
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 51ms/step - accuracy: 0.7815 - loss: 0.4589 - val_accuracy: 0.7835 - val_loss: 0.4675
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7950 - loss: 0.4437 - 

## Part 4 - Making a single prediction

- The model is trained in batches of 32 images, so we need to add a batch dimension to the single image we want to predict.

In [14]:
import numpy as np
from tensorflow.keras.preprocessing import image
from PIL import Image

path = 'dataset/single_prediction/cat_or_dog_2.png'
with Image.open(path) as img:
    has_transparency = (
        'A' in img.getbands() or 'transparency' in img.info
    )
    if has_transparency:
        rgba = img.convert('RGBA')
        background = Image.new('RGBA', rgba.size, 'white')
        img = Image.alpha_composite(background, rgba).convert('RGB')
    else:
        img = img.convert('RGB')
    test_image = img.resize((64, 64))

test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis=0)

result = cnn.predict(test_image)
print(training_set.class_indices)
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
{'cats': 0, 'dogs': 1}


In [15]:
print(prediction)

cat
